In [1]:
import os
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report

matplotlib.use('Agg')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
print(gpus)

[]


In [ ]:
model = tf.keras.models.load_model("/home/admins/lip_codebase_clean/models/model2811_36_21_d130.h5")
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 200704)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │    51,380,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,434,060 (196.21 MB)

 Trainable params: 51,434,058 (196.21 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [9]:
word_classes = ['bat', 'cup', 'drop', 'eat', 'fish', 'hot', 'jump', 'milk', 'pen', 'red']

In [ ]:
IDG = ImageDataGenerator(rescale=1./255)

test = IDG.flow_from_directory(
    "/home/admins/lip_codebase_clean/data/lip_images/7_final_images_130/Test",
    target_size=(224, 224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False,
    seed=42
)

print(test.class_indices)

Found 200 images belonging to 10 classes.


{'bat': 0, 'cup': 1, 'drop': 2, 'eat': 3, 'fish': 4, 'hot': 5, 'jump': 6, 'milk': 7, 'pen': 8, 'red': 9}


In [5]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

loss, acc = model.evaluate(test, verbose=1)
print(f"\nTest loss:     {loss:.4f}")
print(f"Test accuracy: {acc:.4f}  ({acc*100:.2f}%)")

/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1787144418.079346    9843 service.cc:148] XLA service 0x7eded4002510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787144418.079392    9843 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2026-08-19 15:00:18.099413: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787144418.134876    9843 cuda_dnn.cc:529] Loaded cuDNN version 92000


 6/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9646 - loss: 0.2083

I0000 00:00:1787144420.528530    9843 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 135ms/step - accuracy: 0.9761 - loss: 0.1582

Test loss:     0.1230
Test accuracy: 0.9850  (98.50%)


In [7]:
test.reset()
y_prob = model.predict(test, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test.classes

print("predictions:", y_pred.shape, "labels:", y_true.shape)
print("agreement:", (y_pred == y_true).mean())

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step
predictions: (200,) labels: (200,)
agreement: 0.985


In [10]:
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=word_classes, columns=word_classes)
cm_df.index.name = "Actual"
cm_df.columns.name = "Predicted"
print(cm_df)

print()
print(classification_report(y_true, y_pred, target_names=word_classes, digits=4))

Predicted  bat  cup  drop  eat  fish  hot  jump  milk  pen  red
Actual                                                         
bat         19    0     0    0     0    0     0     0    0    1
cup          0   20     0    0     0    0     0     0    0    0
drop         0    0    19    0     0    0     1     0    0    0
eat          0    0     0   20     0    0     0     0    0    0
fish         0    0     0    0    20    0     0     0    0    0
hot          0    0     0    0     0   20     0     0    0    0
jump         0    0     0    0     0    0    20     0    0    0
milk         0    0     0    0     0    0     0    20    0    0
pen          0    0     0    0     0    0     0     1   19    0
red          0    0     0    0     0    0     0     0    0   20

              precision    recall  f1-score   support

         bat     1.0000    0.9500    0.9744        20
         cup     1.0000    1.0000    1.0000        20
        drop     1.0000    0.9500    0.9744        20
         eat  

In [ ]:
os.makedirs("/home/admins/lip_codebase_clean/docs/results_existing_data", exist_ok=True)
out = "/home/admins/lip_codebase_clean/docs/results_existing_data/"

cm_df.to_csv(out + "confusion_matrix.csv")

rep = classification_report(y_true, y_pred, target_names=word_classes,
                            digits=4, output_dict=True)
pd.DataFrame(rep).transpose().to_csv(out + "classification_report.csv")

np.save(out + "y_true.npy", y_true)
np.save(out + "y_pred.npy", y_pred)
np.save(out + "y_prob.npy", y_prob)

with open(out + "summary.txt", "w") as f:
    f.write("Model: model2811_361_21_d130_GOOD.h5\n")
    f.write("Dataset: 7_final_images_130, index-based split\n")
    f.write("Test loss: 0.1230\n")
    f.write("Test accuracy: 0.9850\n")
    f.write("Macro F1: 0.9850\n")
    f.write("Errors: bat->red, drop->jump, pen->milk\n")
    f.write("Environment: TF 2.18.0, Keras 3.7.0, WSL2\n")

print("saved")

saved


In [14]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True,
            square=True, linewidths=0.5, linecolor='lightgray', ax=ax)
ax.set_title('Confusion Matrix — Index-Based Split (Test Accuracy 98.50%)')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(out + 'confusion_matrix.png', dpi=200)
plt.close()

print("saved:", out + 'confusion_matrix.png')

saved: /home/admins/lip_codebase_clean/docs/results_index_split/confusion_matrix.png
